# FIX1 Execution Notebook — simple-permissions

This notebook turns the audit into an execution workflow for the first stabilization pass.

## Scope
- P0 publish/readiness blockers
- Core API safety blockers (`dart:io` web breakage, init guard)
- Stale Android test cleanup

## Working Rules
- Always run status checks before editing.
- Keep changes narrowly scoped per checkpoint.
- Update the checklist cell after each merged fix.


In [ ]:
from pathlib import Path
import subprocess
import json

ROOT = Path.cwd().resolve()
if ROOT.name != 'simple-permissions':
    # If launched from docs/, move to repository root assumptions manually.
    candidate = ROOT
    while candidate != candidate.parent and candidate.name != 'simple-permissions':
        candidate = candidate.parent
    ROOT = candidate

assert ROOT.name == 'simple-permissions', f'Expected simple-permissions root, got: {ROOT}'
print('Repo root:', ROOT)

def sh(*cmd: str) -> str:
    p = subprocess.run(cmd, cwd=ROOT, capture_output=True, text=True)
    out = p.stdout.strip()
    err = p.stderr.strip()
    if p.returncode != 0:
        raise RuntimeError(f'Command failed ({p.returncode}): {' '.join(cmd)}\n{err or out}')
    return out


## Checkpoint 0 — Baseline Snapshot
Run this first in each session to see what changed and avoid clobbering in-progress work.

In [ ]:
print(sh('git', 'status', '--short'))
print('\nTracked target files present:')
targets = [
    'README.md',
    'CHANGELOG.md',
    'LICENSE',
    'pubspec.yaml',
    'lib/simple_permissions.dart',
    'lib/src/intention.dart',
    'pigeon.dart',
    'android/src/main/kotlin/io/simplezen/simple_permissions/PermissionsHostApiImpl.kt',
    'android/src/test/kotlin/com/example/simple_permissions/SimplePermissionsPluginTest.kt',
]
for rel in targets:
    print(f'- {rel}:', 'OK' if (ROOT / rel).exists() else 'MISSING')


## Checkpoint 1 — P0 Task Board
Update statuses after each change so the notebook can serve as the implementation log.

In [ ]:
P0_TASKS = [
    {'id': 'P0-1', 'title': 'Replace README/CHANGELOG/LICENSE placeholders', 'status': 'todo', 'files': ['README.md', 'CHANGELOG.md', 'LICENSE']},
    {'id': 'P0-2', 'title': 'Remove dart:io-dependent platform checks from public API path', 'status': 'todo', 'files': ['lib/simple_permissions.dart']},
    {'id': 'P0-3', 'title': 'Add initialization guard with actionable error', 'status': 'todo', 'files': ['lib/simple_permissions.dart']},
    {'id': 'P0-4', 'title': 'Remove or replace stale Kotlin template test', 'status': 'todo', 'files': ['android/src/test/kotlin/com/example/simple_permissions/SimplePermissionsPluginTest.kt']},
]

for t in P0_TASKS:
    print(f"[{t['status']}] {t['id']} - {t['title']}")
    print('  files:', ', '.join(t['files']))


## Checkpoint 2 — Automated P0 Readiness Checks
This cell highlights current blockers from the audit so fixes are verifiable.

In [ ]:
def read(rel: str) -> str:
    return (ROOT / rel).read_text(encoding='utf-8')

issues = []

# Placeholder docs checks
readme = read('README.md')
if 'A new Flutter plugin project.' in readme:
    issues.append('README.md still has template copy')

changelog = read('CHANGELOG.md')
if 'TODO: Describe initial release.' in changelog:
    issues.append('CHANGELOG.md still has TODO placeholder')

license_text = read('LICENSE')
if 'TODO: Add your license here.' in license_text:
    issues.append('LICENSE is still placeholder')

# API safety checks
sp = read('lib/simple_permissions.dart')
if "import 'dart:io';" in sp:
    issues.append('simple_permissions.dart imports dart:io (web-incompatible)')
if '_hostApi!' in sp and '_ensureInitialized' not in sp:
    issues.append('simple_permissions.dart has force-unwrapped _hostApi without explicit guard helper')

# Stale Kotlin template test check
kt_test = read('android/src/test/kotlin/com/example/simple_permissions/SimplePermissionsPluginTest.kt')
if 'onMethodCall' in kt_test:
    issues.append('Android unit test still references onMethodCall template API')

print('Blocking issues found:' if issues else 'No blocking issues detected by this check set.')
for i, issue in enumerate(issues, start=1):
    print(f'{i}. {issue}')


## Checkpoint 3 — Quick API Contract Probe
Use this to confirm the public API is moving toward intention-first ergonomics.

In [ ]:
sp = read('lib/simple_permissions.dart')
probes = {
    'has_check_permissions_list_api': 'checkPermissions(List<String> permissions)' in sp,
    'has_request_permissions_list_api': 'requestPermissions(List<String> permissions)' in sp,
    'has_intention_check_wrapper': 'check(Intention intention)' in sp,
    'has_intention_request_wrapper': 'request(Intention intention)' in sp,
}
print(json.dumps(probes, indent=2))


## Next Build Targets
1. Land P0 cleanups (docs/license/changelog + API guard + stale test).
2. Add intention-level wrappers with richer result model (P1).
3. Extend Pigeon API for rationale/settings methods.

After each merged step, rerun Checkpoints 0-3 and capture outputs in git history.

# Audit Report: simple-permissions

**Files Reviewed**: 16 (all Dart, Kotlin, YAML, XML, Markdown, and Gradle files)
**Issues Found**: 22
**Severity Breakdown**: :red_circle: Critical: 3 | :orange_circle: High: 7 | :yellow_circle: Medium: 7 | :large_blue_circle: Low: 5

---

## :red_circle: Critical Issues

### [C-1] No iOS platform implementation — description claims cross-platform readiness

- **File**: pubspec.yaml
- **Category**: Architecture / pub.dev Readiness
- **Issue**: The pubspec.yaml `flutter.plugin.platforms` block **only** declares android. There is no ios directory, no Swift/ObjC implementation, and no Dart platform-interface fallback. Yet the library description says *"Unified permission handling for Flutter apps"* and the Dart code silently returns `true` for non-Android — **pretending permissions are granted when they are not**.
- **Why it matters**: Any consumer running on iOS, web, macOS, or Linux will silently get `true` for every permission check, which is dangerously misleading. For pub.dev listing, the package will show "Android only" in the platform badges while giving a false sense of cross-platform safety at runtime.
- **Evidence** (simple_permissions.dart):
  ```dart
  Future<Map<String, bool>> checkPermissions(List<String> permissions) async {
    if (!Platform.isAndroid) {
      return {for (var p in permissions) p: true};
    }
  ```
  Returning `true` silently is the worst possible default — consumers will skip permission flows and hit hard crashes or silent data loss on unsupported platforms.

### [C-2] `dart:io` import makes the library unusable on web

- **File**: simple_permissions.dart
- **Category**: Architecture / Platform Compatibility
- **Issue**: `import 'dart:io';` is used solely for `Platform.isAndroid` checks. `dart:io` is not available on web, so this package will cause **compile-time failures** for any Flutter Web project that depends on it — even transitively.
- **Why it matters**: pub.dev static analysis will flag this. The Flutter-idiomatic approach is to use `defaultTargetPlatform` from `package:flutter/foundation.dart`, or use a platform interface / conditional imports.
- **Evidence**:
  ```dart
  import 'dart:io';
  // ...
  if (!Platform.isAndroid) {
  ```

### [C-3] `_hostApi` is force-unwrapped without initialization guard

- **File**: simple_permissions.dart
- **Category**: Robustness / Error Handling
- **Issue**: Every method accesses `_hostApi!` without checking `_initialized`. If any consumer calls `instance.checkPermissions(...)` before `initialize()`, they'll get an unhelpful `Null check operator used on a null value` exception on Android.
- **Why it matters**: This is the #1 issue users open on pub.dev plugins. A clear `StateError('SimplePermissions not initialized. Call SimplePermissions.initialize() first.')` would save hours of debugging.
- **Evidence**:
  ```dart
  static PermissionsHostApi? _hostApi;
  // ...
  Future<Map<String, bool>> checkPermissions(List<String> permissions) async {
    if (!Platform.isAndroid) { ... }
    final result = await _hostApi!.checkPermissions(permissions); // 💥
  ```

---

## :orange_circle: High Severity

### [H-1] LICENSE file is a placeholder

- **File**: LICENSE
- **Category**: pub.dev Readiness
- **Issue**: Contains only `TODO: Add your license here.` Pub.dev **requires** a valid license. The `pana` scoring tool will give 0 points for the license category and the package cannot be published.
- **Why it matters**: Blocks publishing entirely. Also legally problematic — without a license, the code is "all rights reserved" by default.

### [H-2] CHANGELOG.md is a placeholder

- **File**: CHANGELOG.md
- **Category**: pub.dev Readiness
- **Issue**: Contains `## 0.0.1 * TODO: Describe initial release.` while the pubspec.yaml declares version `0.1.0`. Version mismatch + placeholder content.
- **Why it matters**: `pana` will flag the version mismatch. Changelog is displayed on the pub.dev package page — placeholder text looks unprofessional and hurts adoption.

### [H-3] README.md is the default template

- **File**: README.md
- **Category**: pub.dev Readiness
- **Issue**: Still contains the Flutter template boilerplate text: *"A new Flutter plugin project."* and *"The plugin project was generated without specifying the --platforms flag"*. No API documentation, no usage examples, no feature list.
- **Why it matters**: The README is the first thing users see on pub.dev. `pana` scores documentation quality. This will lose significant points and deter adoption.

### [H-4] Kotlin test is a stale template that won't compile

- **File**: SimplePermissionsPluginTest.kt
- **Category**: Testing / Code Quality
- **Issue**: The test calls `plugin.onMethodCall(call, mockResult)` — but `SimplePermissionsPlugin` does **not** implement `MethodCallHandler` and has no `onMethodCall` method. This is the auto-generated template test from `flutter create` that was never updated. It will fail to compile.
- **Why it matters**: Dead test code that gives a false sense of coverage. The package path `com.example.simple_permissions` also doesn't match the real package `io.simplezen.simple_permissions`.

### [H-5] No error handling / custom exceptions on the Dart side

- **File**: simple_permissions.dart
- **Category**: Error Handling / API Design
- **Issue**: All methods simply let `PlatformException` propagate raw from Pigeon. There are no custom exception types, no wrapping, no documentation of what errors to expect. If the Activity is detached, the user sees `PlatformException(java.lang.Exception, Activity detached)` with no context.
- **Why it matters**: A pub.dev-quality plugin should define clear error types (e.g., `PermissionException`, `NoActivityException`) so consumers can catch and handle gracefully.

### [H-6] Race condition: concurrent async requests will clobber callbacks

- **File**: PermissionsHostApiImpl.kt
- **Category**: Concurrency / Correctness
- **Issue**: `pendingPermissionsCallback`, `pendingRoleCallback`, and `pendingBatteryCallback` are single-slot fields. If `requestPermissions()` is called twice before the first returns, the first callback is silently overwritten and **never invoked** — leaving a Dart `Future` hanging forever.
- **Why it matters**: In real apps, rapid UI interactions or multiple permission screens can easily trigger concurrent requests. A dangling Future means the app freezes from the user's perspective.
- **Evidence**:
  ```kotlin
  pendingPermissionsCallback = callback  // overwrites previous without invoking it
  pendingPermissions = permissions.toTypedArray()
  ```

### [H-7] `plugin_platform_interface` dependency is unused

- **File**: pubspec.yaml
- **Category**: Code Quality / pub.dev Readiness
- **Issue**: `plugin_platform_interface: ^2.0.2` is declared as a dependency but never used anywhere in the codebase. There is no platform interface class, no method channel implementation class, and no federated plugin structure.
- **Why it matters**: Unused dependencies increase install size and confuse contributors. `pana` and `dart analyze` may flag this. More importantly, for pub.dev listing a proper platform interface is the recommended architecture for plugins.

---

## :yellow_circle: Medium Severity

### [M-1] `Intention` enum uses hardcoded Android permission strings — not portable

- **File**: intention.dart
- **Category**: Architecture / Design
- **Issue**: The `Intention` enum hardcodes Android-specific permission strings (`android.permission.SEND_SMS`) and role strings (`android.app.role.SMS`). This makes the abstraction layer Android-only by design — if iOS support is added later, this entire enum design needs rethinking.
- **Why it matters**: The whole point of an `Intention`-based API is platform abstraction, but the implementation leaks platform details. Consider making `permissions` return platform-appropriate values.

### [M-2] `fileAccess` intention mixes deprecated and modern permissions

- **File**: intention.dart
- **Category**: Correctness
- **Issue**: `READ_EXTERNAL_STORAGE` is largely deprecated on API 33+ (the same target where `READ_MEDIA_*` were introduced). Requesting both simultaneously will cause confusing results — `READ_EXTERNAL_STORAGE` will always be denied on API 33+ even if `READ_MEDIA_*` are granted.
- **Why it matters**: Consumers will see mixed `true`/`false` results and think something is broken. The intention should be API-level-aware, or document this clearly.
- **Evidence**:
  ```dart
  case Intention.fileAccess:
    return const [
      'android.permission.READ_EXTERNAL_STORAGE',  // deprecated API 33+
      'android.permission.READ_MEDIA_IMAGES',       // added API 33
      'android.permission.READ_MEDIA_VIDEO',
      'android.permission.READ_MEDIA_AUDIO',
    ];
  ```

### [M-3] `initialize()` is `async` but does nothing async

- **File**: simple_permissions.dart
- **Category**: API Design
- **Issue**: `initialize()` is declared `Future<void>` and uses `async`, but the body is entirely synchronous. This forces consumers to `await` for no reason.
- **Why it matters**: Minor API friction, but more importantly it sets an erroneous expectation of heavy initialization. If you plan to add truly async init later (e.g., checking initial permission state), the signature is fine — but document that intent.

### [M-4] `onDetachedFromActivity` removes listener from wrong binding

- **File**: PermissionsHostApiImpl.kt
- **Category**: Correctness
- **Issue**: `onDetachedFromActivity()` calls `activityBindingProvider()?.removeActivityResultListener(this)`. But by the time `onDetachedFromActivity` is called from the plugin, the plugin **has already nulled** `activityBinding`. So `activityBindingProvider()` will return `null` and the listener is **never removed**.
- **Why it matters**: Leaked `ActivityResultListener` references. The listener was added in `onAttachedToActivity` but the corresponding removal is effectively a no-op.
- **Evidence** in SimplePermissionsPlugin.kt:
  ```kotlin
  override fun onDetachedFromActivity() {
      activityBinding?.removeRequestPermissionsResultListener(this)
      permissionsHostApiImpl?.onDetachedFromActivity()  // called AFTER binding = null
      activityBinding = null  // already null by sequence? No — this happens after.
      activity = null
  }
  ```
  Actually, re-reading: the plugin calls `impl.onDetachedFromActivity()` **before** setting `activityBinding = null`. But `PermissionsHostApiImpl.onDetachedFromActivity()` calls `activityBindingProvider()` which is `{ activityBinding }` — the plugin's field is still set at that point. So this is **correct** for `onDetachedFromActivity`, BUT for `onDetachedFromActivityForConfigChanges` the sequence is the same. Let me downgrade: the ordering is actually fine. However, the impl adds itself as a listener in `onAttachedToActivity` but the plugin's `onDetachedFromActivity` only removes the **plugin** (as `RequestPermissionsResultListener`), NOT the impl (as `ActivityResultListener`). The impl self-removes in its own `onDetachedFromActivity`, which is fine.

  **Revised**: The logic is sound upon closer reading. Withdrawing this finding — the call order is correct. *(Self-correction: no issue here.)*

### [M-4] (Revised) No `shouldShowRequestPermissionRationale` support

- **File**: PermissionsHostApiImpl.kt
- **Category**: Missing Feature / Best Practice
- **Issue**: The plugin provides no way to check `shouldShowRequestPermissionRationale()`. This is a key part of Android's permission UX: after the user denies once, the app should explain why the permission is needed before asking again.
- **Why it matters**: Without this, consumers can't build proper permission flows. Every popular permission plugin (permission_handler, etc.) includes this.

### [M-5] No `openAppSettings()` method

- **File**: Plugin-wide
- **Category**: Missing Feature
- **Issue**: There's no way to direct users to the app's settings page when permissions are permanently denied. This is table-stakes for a permission plugin.
- **Why it matters**: On Android, after "Don't ask again" is selected, the only way to grant a permission is through Settings. Without this method, the plugin is incomplete for production use.

### [M-6] Kotlin version `1.8.22` is significantly outdated

- **File**: build.gradle
- **Category**: Maintenance / Compatibility
- **Issue**: `ext.kotlin_version = "1.8.22"` — current Kotlin is 2.x, Flutter's recommended minimum is 1.9+. The `buildscript` block with explicit `classpath` dependencies is the legacy Gradle setup; most modern Flutter plugins use the `plugins {}` block.
- **Why it matters**: May cause compatibility issues with consumers using newer Kotlin versions. AGP 8.7.0 + Kotlin 1.8.22 is a tested-but-aging combination.

### [M-7] .gitignore excludes pubspec.lock and .github

- **File**: .gitignore
- **Issue**: `*.lock` is in the gitignore, which is correct for a library (pub.dev recommends NOT committing pubspec.lock for packages). However, .github is also ignored — meaning CI configs, copilot instructions, etc. won't be version-controlled. This is likely unintentional.

---

## :large_blue_circle: Low Severity / Nitpicks

### [L-1] `homepage` URL likely doesn't exist

- **File**: pubspec.yaml
- **Issue**: `homepage: https://github.com/simplezen/simple-permissions` — verify this URL exists. `pana` will HTTP-check it and deduct points if it 404s. Consider also adding `repository:` and `issue_tracker:` fields.

### [L-2] Example app's `_checkPermissions` uses `!` on nullable role

- **File**: example/lib/main.dart
- **Issue**: `Intention.texting.role!` — While `texting.role` is always non-null, if someone copy-pastes this pattern for `contacts` or `notifications` it will crash. The example should demonstrate safe handling.

### [L-3] `description` in pubspec.yaml is under 60 chars but could be richer

- **File**: pubspec.yaml
- **Issue**: pub.dev recommends 60-180 characters for the description. Current is adequate but generic. Missing keywords like "Android permissions", "runtime permissions", "role manager" that would improve discoverability.

### [L-4] Version mismatch: pubspec says `0.1.0`, changelog says `0.0.1`

- **File**: pubspec.yaml vs CHANGELOG.md
- **Issue**: Minor version discrepancy. pub.dev expects these to be consistent.

### [L-5] Unused import in generated code: `dart:async`

- **File**: permissions.g.dart
- **Issue**: Pigeon generates `import 'dart:async';` which is unused. Not actionable (autogenerated), but `dart analyze` with strict settings may flag it.

---

## Patterns & Trends

1. **Template residue everywhere**: LICENSE, CHANGELOG, README, Kotlin test — all are unchanged from `flutter create`. This is the single biggest blocker for pub.dev.
2. **Android-only with false cross-platform facade**: The `Platform.isAndroid` checks silently succeed on other platforms instead of throwing `UnsupportedError`. This is a design choice that will bite consumers.
3. **Missing defensive programming**: No init guards, no concurrent-request protection, no custom exceptions. The happy path works; anything off the rails gives cryptic errors.
4. **Missing standard permission plugin features**: No `shouldShowRationale`, no `openAppSettings()`, no permission status enum (granted/denied/permanentlyDenied/restricted). Competing packages like `permission_handler` set the bar here.

---

## Top 3 Recommendations

1. **Replace all template placeholders (LICENSE, CHANGELOG, README)**: This is the fastest path to unblocking a pub.dev publish. Write a real README with API examples (the code for this already exists in your doc comments), pick a license (MIT/BSD-3 are standard for Flutter plugins), and sync the changelog version.

2. **Replace `dart:io` with `defaultTargetPlatform` and add an uninitialized guard**: These two changes fix C-2 and C-3 with minimal effort. Optionally throw `UnsupportedError` on non-Android instead of returning `true`.

3. **Add `shouldShowRequestPermissionRationale()` and `openAppSettings()`**: These two methods are the minimum bar to compete with existing permission plugins. Without them, consumers will immediately reach for `permission_handler` instead — and there goes your pub.dev niche.

## Addendum — Latest Findings (Feb 24, 2026)

### Scope of this update
- Cross-checked the current implementation against `docs/PROJECT_WHITEPAPER.md`.
- Evaluated readiness for **Android 31+** specifically.
- Evaluated whether the API is truly **"extremely easy to use"** for Dart consumers.

---

## 1) Whitepaper vs Implementation: Current Reality

### Architecture divergence (important)
- The whitepaper describes a **federated plugin** (`SimplePermissionsPlatform`, `MethodChannelSimplePermissions`, token verification).
- Current implementation uses **Pigeon-generated host APIs** directly and does not implement that federated layer.
- This is not inherently bad (Pigeon gives strong typing), but docs are now materially out of date.

### API shape divergence (high DX impact)
- Whitepaper promises an intention-first API:
  - `request(Intention)`
  - `check(Intention)`
  - `PermissionResult`
  - `PermissionStatus` (including `permanentlyDenied`)
- Current public API is lower-level:
  - `requestPermissions(List<String>)`
  - `checkPermissions(List<String>)`
  - role calls are separate (`isRoleHeld`, `requestRole`)
- Result: consumers must orchestrate role + permissions manually for common flows.

### Documentation drift
- Whitepaper still claims native Android implementation pending, but Android native implementation exists.
- Whitepaper references `getPlatformVersion` and MethodChannel contracts that are no longer the source of truth.
- Compatibility tables in the whitepaper do not match current build settings and implementation details.

---

## 2) Android 31+ Readiness Assessment

### What is good
- `compileSdk = 36` is current and future-friendly.
- Runtime permission handling and role flows are present.
- Notification permission (`POST_NOTIFICATIONS`) intention exists for Android 13+.

### Gaps for a strong 31+ positioning
- No explicit support for **rationale/permanent denial** classification (`shouldShowRequestPermissionRationale` not exposed).
- No **open app settings** helper for the permanent-denial recovery path.
- `fileAccess` combines legacy + modern media permissions; behavior differs across API levels and should be normalized in API docs + return model.
- Android 14+ partial media access nuances are not represented in the result model.

### Policy/behavior caveat
- Returning implicit success on unsupported platforms (non-Android) remains risky and can hide incorrect app behavior.

---

## 3) Ease-of-Use Assessment ("Extremely Easy" goal)

### Current friction points
- Requires explicit `initialize()` and multiple method calls for standard workflows.
- No one-call “ensure this intention is ready” API.
- `Map<String, bool>` return type is too primitive for real permission UX.
- No first-class modeling of:
  - denied vs permanently denied
  - role denied vs permission denied
  - next-step guidance (retry, rationale, settings)

### Minimal target API for excellent DX
- `check(Intention intention)` → rich result
- `request(Intention intention)` → rich result
- `openAppSettings()`
- `shouldShowRationale(Intention intention)`

A rich result should include:
- per-permission status
- role status
- aggregate booleans (`allGranted`, `isFullyGranted`)
- convenience flags (`hasPermanentDenial`, `requiresSettings`) 

---

## 4) Prioritized Action Plan (Android 31+ + pub.dev readiness)

### P0 — Must do now
1. Replace placeholder docs/artifacts:
   - `README.md`, `CHANGELOG.md`, `LICENSE`
2. Remove web-hostile platform check approach (`dart:io` usage in public API path).
3. Add initialization guard with a clear developer-facing error.
4. Fix/remove stale Kotlin template test that references nonexistent APIs.

### P1 — Needed for “extremely easy” API
1. Add intention-level convenience API (`request(Intention)`, `check(Intention)`).
2. Introduce `PermissionStatus` + `PermissionResult` model.
3. Add `openAppSettings()`.
4. Add rationale support (`shouldShowRationale`).

### P2 — Android 31+ polish
1. Normalize media/storage handling by API level.
2. Improve result semantics for Android 13/14 behaviors.
3. Add test matrix notes for API 31, 33, and 34+.

---

## 5) Updated Severity Snapshot (with latest findings)

- **Critical**: Core DX contract mismatch with stated product goal (intention-first simplicity not delivered).
- **High**: Publish blockers (license/changelog/readme), stale tests, platform behavior ambiguity.
- **Medium**: Android-version-specific semantics (media/rationale/settings) not represented in API model.
- **Low**: Documentation/version drift and cleanup items.

---

## 6) Recommended Definition of Done for next milestone

- Public API supports one-call intention checks/requests.
- Permanent denial and rationale are first-class in return types.
- Android 31/33/34 behavior is documented and tested.
- README demonstrates a complete real-world flow in <20 lines.
- `flutter analyze`, `flutter test`, and `flutter pub publish --dry-run` pass cleanly.

---

## 7) Decision record for this audit iteration

- Keep Pigeon architecture (acceptable and type-safe), but update docs to match reality.
- Optimize for Android 31+ first; iOS expansion can follow as a separate milestone.
- Prioritize developer ergonomics (intention-level API) over adding many new intentions.


## Engineering Checklist — Execution Ready

### Goal
Deliver an Android 31+ friendly, pub.dev-ready, intention-first permissions plugin with minimal friction for app developers.

---

### P0 — Publish & correctness blockers

#### 1) Replace placeholder package docs
- **Files**:
  - `README.md`
  - `CHANGELOG.md`
  - `LICENSE`
- **Tasks**:
  - Replace template README with real usage, API reference, platform support matrix, and known limitations.
  - Align changelog version with `pubspec.yaml` and document current release contents.
  - Add an OSS license (MIT/BSD-3/Apache-2.0).
- **Done when**:
  - No placeholder text remains.
  - `flutter pub publish --dry-run` no longer flags missing quality artifacts.

#### 2) Fix platform detection path in Dart API
- **Files**:
  - `lib/simple_permissions.dart`
- **Tasks**:
  - Remove `dart:io` dependency from public runtime path.
  - Use Flutter-safe platform detection approach.
  - Stop returning implicit success for unsupported platforms (return explicit unsupported error or clearly documented behavior).
- **Done when**:
  - Package compiles cleanly for Flutter targets without `dart:io` runtime assumptions.
  - Unsupported platform behavior is explicit and documented.

#### 3) Add initialization guardrails
- **Files**:
  - `lib/simple_permissions.dart`
- **Tasks**:
  - Guard all host API calls behind initialization checks.
  - Throw clear actionable error message if API is used before init.
- **Done when**:
  - Calling any API before init yields deterministic developer-friendly error.

#### 4) Remove/replace stale Kotlin template test
- **Files**:
  - `android/src/test/kotlin/com/example/simple_permissions/SimplePermissionsPluginTest.kt`
- **Tasks**:
  - Delete invalid template test or replace with real tests for `PermissionsHostApiImpl` behavior.
- **Done when**:
  - Android unit tests compile and run without referencing non-existent plugin APIs.

---

### P1 — “Extremely easy to use” API milestone

#### 5) Introduce intention-first convenience API
- **Files**:
  - `lib/simple_permissions.dart`
  - `lib/src/intention.dart`
- **Tasks**:
  - Add `check(Intention intention)` and `request(Intention intention)` wrappers.
  - Internally orchestrate role + runtime permission flow in correct order.
- **Done when**:
  - Common SMS/dialer flows can be completed with one high-level call.

#### 6) Add rich result model
- **Files**:
  - `lib/src/` (new model file(s), or in `simple_permissions.dart`)
- **Tasks**:
  - Add `PermissionStatus` enum and `PermissionResult` DTO.
  - Include role status, per-permission status, aggregate flags.
- **Done when**:
  - Consumers can distinguish denied vs permanently denied and role failures.

#### 7) Add settings recovery API
- **Files**:
  - `pigeon.dart`
  - `lib/src/generated/permissions.g.dart` (generated)
  - `android/src/main/kotlin/io/simplezen/simple_permissions/Permissions.g.kt` (generated)
  - `android/src/main/kotlin/io/simplezen/simple_permissions/PermissionsHostApiImpl.kt`
- **Tasks**:
  - Add `openAppSettings()` host call.
  - Implement Android native intent to app settings page.
- **Done when**:
  - App can deep-link users to settings after permanent denial.

#### 8) Add rationale support
- **Files**:
  - `pigeon.dart`
  - generated bindings (Dart/Kotlin)
  - `PermissionsHostApiImpl.kt`
- **Tasks**:
  - Add method to report `shouldShowRequestPermissionRationale` for one or multiple permissions.
- **Done when**:
  - Caller can build correct pre-permission rationale UX.

---

### P2 — Android 31+ semantic polish

#### 9) Normalize media/storage behavior by API level
- **Files**:
  - `lib/src/intention.dart`
  - `README.md`
- **Tasks**:
  - Clarify version-specific behavior for `READ_EXTERNAL_STORAGE` vs `READ_MEDIA_*`.
  - Ensure returned statuses are understandable on API 33+.
- **Done when**:
  - No confusing mixed outcomes for file access without documentation.

#### 10) Capture Android 31/33/34 test matrix
- **Files**:
  - `README.md`
  - `example/integration_test/plugin_integration_test.dart`
  - `test/simple_permissions_test.dart`
- **Tasks**:
  - Add test scenarios for API 31 baseline, API 33 notification/media behavior, API 34+ media nuances.
- **Done when**:
  - Reproducible test matrix exists in docs/tests.

---

### Concurrency hardening checklist

#### 11) Prevent in-flight callback clobbering
- **Files**:
  - `android/src/main/kotlin/io/simplezen/simple_permissions/PermissionsHostApiImpl.kt`
- **Tasks**:
  - Reject or queue concurrent requests per operation type.
  - Return explicit error when a second request is attempted while one is in progress.
- **Done when**:
  - No request path can leave a Dart `Future` unresolved due to overwritten callbacks.

---

### Quality gate checklist (must pass before publish)

- [ ] `flutter analyze` passes
- [ ] `flutter test` passes
- [ ] Android unit tests compile and pass
- [ ] `flutter pub publish --dry-run` passes with no critical quality warnings
- [ ] README examples compile against the current API
- [ ] Platform support table and behavior notes match actual implementation

---

### Suggested implementation order (fastest risk burn-down)
1. P0 docs/license/changelog + stale test cleanup
2. Init guard + platform detection fix
3. Intention-first `check/request` wrappers
4. Rich result model
5. Settings + rationale API additions via Pigeon
6. Concurrency hardening
7. Android 31/33/34 test matrix + final docs sync